In [1]:
from typing import Literal
import torch

device: Literal["cuda", "cpu"] = "cuda" if torch.cuda.is_available() else "cpu"
if(device == "cuda"):
    print(torch.cuda.get_device_name())
else:
    print("cpu")

NVIDIA GeForce MX350


In [2]:
from torchvision.transforms import v2
from torchvision.datasets import FashionMNIST
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
    def __init__(self, mode: str="train", transforms=None):
        self.dataset: FashionMNIST = FashionMNIST(
            root=f"./data/{mode}/",
            train=True if mode == "train" else False,
            transform=None,
            target_transform=None,
            download=True
        )

        self.transforms = transforms
        if(mode == "train" and self.transforms is None):
            self.transforms = v2.Compose(transforms=[
                v2.RandomHorizontalFlip(p=0.5),
                v2.RandomRotation(degrees=90),
                v2.PILToTensor(),
                v2.ToDtype(dtype=torch.float32, scale=False)
            ])
        else:
            self.transforms = v2.Compose(transforms=[
                v2.PILToTensor(),
                v2.ToDtype(dtype=torch.float32, scale=False)
            ])

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        image, lable = self.dataset[index] # self.dataset[index] returns an (image, label) pair
        image = self.transforms(image) # shape - [1, 28, 28]
        return image, lable

train_dataset = CustomDataset(mode="train", transforms=None)
test_dataset = CustomDataset(mode="test", transforms=None)

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=64,
    shuffle=True,
    pin_memory=True,
    drop_last=False
)

test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=64,
    shuffle=True,
    pin_memory=True,
    drop_last=False
)

In [3]:
import torch.nn as nn

class CustomModel(nn.Module):
    def __init__(self, in_channels: int=3, num_classes: int=10):
        super().__init__()
        self.convolution = nn.Sequential(
            nn.Conv2d(in_channels=in_channels, out_channels=10, kernel_size=(10, 10), stride=1),
            nn.MaxPool2d(kernel_size=(3, 3), stride=1),

            nn.Conv2d(in_channels=10, out_channels=5, kernel_size=(5, 5), stride=1),
            nn.MaxPool2d(kernel_size=(3, 3), stride=1),

            nn.Conv2d(in_channels=5, out_channels=3, kernel_size=(3, 3), stride=1),
            nn.MaxPool2d(kernel_size=(3, 3), stride=1),
        )

        self.linear = nn.Sequential(
            nn.Flatten(start_dim=1, end_dim=-1),

            nn.Linear(in_features=147, out_features=64),
            nn.GELU(),

            nn.Linear(in_features=64, out_features=num_classes)
        )
    
    def forward(self, input):
        convolution_output = self.convolution(input)
        linear_output = self.linear(convolution_output)
        return convolution_output, linear_output

with torch.no_grad():
    model = CustomModel(in_channels=1, num_classes=10).to(device=device)
    colvolution_output, linear_output = model((test_dataset[1][0]).unsqueeze(dim=0).to(device))
    print(linear_output)

tensor([[ 5.3397,  5.9965,  4.0371,  3.5864, -5.2303,  4.6494, -0.3456, -7.9216,
         -3.4314,  1.3249]], device='cuda:0')


In [ ]:
epochs: int = 10
learning_rate: float = 0.001

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    params=model.parameters(),
    lr=learning_rate
)

print("Model Training")
model.train(mode=True)
epoch_loss: list[float] = []
for i in range(10):
    total_loss: float = 0.0
    batch_images: int = 0 # NOTE: Each batch does not have complete 64 images

    for image, label in train_dataloader:
        optimizer.zero_grad()
        image, label = image.to(device=device), label.to(device=device)

        colvolution_output, linear_output = model(image)
        loss = criterion(linear_output, label)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        batch_images += linear_output.shape[0]

    total_loss = total_loss / (len(train_dataloader) * batch_images)
    epoch_loss.append(total_loss)

    print(f"Epoch {i + 1} Loss: {total_loss}")

Model Training
Epoch 1 Loss: 2.0615645271925192e-05
Epoch 2 Loss: 1.682224268386274e-05
Epoch 3 Loss: 1.5778535150134486e-05
Epoch 4 Loss: 1.5220137939185392e-05
Epoch 5 Loss: 1.4838755974849234e-05
Epoch 6 Loss: 1.458755720462372e-05
Epoch 7 Loss: 1.4365302087168535e-05
Epoch 8 Loss: 1.4182439950568166e-05
Epoch 9 Loss: 1.418593672780001e-05
Epoch 10 Loss: 1.399505615668595e-05
Model Evaluation
Training Accuract: {accuracy}
Testing Accuract: {accuracy}


In [6]:
print("Model Evaluation")
model.eval()
with torch.no_grad():
    total_correct: int = 0
    total: int = 0
    for image, label in train_dataloader:
        optimizer.zero_grad()
        image, label = image.to(device=device), label.to(device=device)

        colvolution_output, linear_output = model(image)
        prediction: torch.Tensor = torch.argmax(input=linear_output, dim=1)

        total_correct += torch.sum(label == prediction).item()
        total += linear_output.shape[0]
    
    accuracy: float = (total_correct / total)
    print(f"Training Accuract: {accuracy}")

    total_correct = 0
    total = 0
    for image, label in test_dataloader:
        optimizer.zero_grad()
        image, label = image.to(device=device), label.to(device=device)

        colvolution_output, linear_output = model(image)
        prediction: torch.Tensor = torch.argmax(input=linear_output, dim=1)

        total_correct += torch.sum(label == prediction).item()
        total += linear_output.shape[0]
    
    accuracy: float = (total_correct / total)
    print(f"Testing Accuract: {accuracy}")

Model Evaluation
Training Accuract: 0.6907666666666666
Testing Accuract: 0.6677


In [5]:
checkpoint = {
    "model": model.state_dict(),
    "optimizer": optimizer.state_dict(),
    "learning_rate": learning_rate,
    "epochs": epochs,
    "loss": criterion,
    "epoch_loss": epoch_loss
}
torch.save(obj=checkpoint, f="./model/model_checkpoint.pth")

checkpoint = torch.load(f="./model/model_checkpoint.pth", weights_only=False)
model.load_state_dict(state_dict=checkpoint["model"])
optimizer.load_state_dict(state_dict=checkpoint["optimizer"])

with torch.no_grad():
    colvolution_output, linear_output = model((test_dataset[1][0]).unsqueeze(dim=0).to(device))
    print(train_dataset.dataset.classes[torch.argmax(linear_output)])
    print(train_dataset.dataset.classes[test_dataset[1][1]])

Pullover
Pullover
